# NLP 传统文本表示：从 One-Hot、BoW 到 TF-IDF 和 N-gram

Embedding（嵌入表示）不一定一开始就指神经网络中的 nn.Embedding。在自然语言处理的经典阶段，人们先把文本转换成可计算的向量，再交给相似度算法、线性模型或浅层神经网络。常见传统表示包括：

- One-Hot：用一个位置为 1、其余位置为 0 的稀疏向量表示 token；
- BoW（Bag of Words）：用词频表示一篇文档，忽略词序；
- N-gram：把连续的 n 个 token 作为特征，保留有限的局部顺序；
- TF-IDF：降低整个语料中普遍出现的词的权重，突出文档特有词；
- 哈希向量、共现矩阵和 LSA/LSI：进一步压缩或变换稀疏表示。

这些方法通常被称为传统文本向量化或传统 embedding。它们与神经网络中的稠密、可训练 embedding 有重要区别：传统方法往往由词频和统计公式直接构造，神经 embedding 则通过训练目标学习参数。本文从 0 到 1 讲清这条演进路线，并在最后比较它们与 PyTorch nn.Embedding、Transformer tokenizer 的边界。

本文示例主要使用 Python、PyTorch、NumPy、scikit-learn 和 jieba。项目中已经声明 NumPy、scikit-learn 和 jieba 依赖；若在独立环境运行，请先安装对应包。

In [19]:
import re
import math
from collections import Counter
import nltk
import jieba

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from typing import List, Tuple
from nltk.corpus import stopwords
import string
import collections

torch.manual_seed(23)

In [26]:
# 0、先完成通用方法 tokenizer、vocab 等

text = """The Time Traveller (for so it will be convenient to speak of him)
was expounding a recondite matter to us. His grey eyes shone and
twinkled, and his usually pale face was flushed and animated. The
fire burned brightly, and the soft radiance of the incandescent
lights in the lilies of silver caught the bubbles that flashed and
passed in our glasses. Our chairs, being his patents, embraced and
caressed us rather than submitted to be sat upon, and there was that
luxurious after-dinner atmosphere when thought roams gracefully
free of the trammels of precision. And he put it to us in this
way--marking the points with a lean forefinger--as we sat and lazily
admired his earnestness over this new paradox (as we thought it)
and his fecundity."""

lines = text.lower().split()
print(f"lines Tokens Len: {len(lines)}, token: {lines[:10]}")


def tokenize(lines: List[str], token: str = "word") -> List[List[str]]:
    if token == "word":
        # .split() 默认以空格分割字符串
        return [line.split() for line in lines]
    if token == "char":
        return [list(line) for line in lines]
    raise ValueError(f"未知 token 类型: {token}")

tokens = tokenize(lines, token="word")
print("tokens", len(src_tokens), src_tokens[:10])



# 去除标点符号
punctuation = list(string.punctuation)
tokens = [token for line in tokens for token in line if token not in punctuation]
print(f"punctuation Tokens Len: {len(tokens)}, token: {tokens[:10]}")

# 去除停留词
english_stopwords = set(stopwords.words("english"))
tokens = [token for token in tokens if token not in english_stopwords]
print(f"stopwords Tokens Len: {len(tokens)}, token: {tokens[:10]}")



lines Tokens Len: 128, token: ['the', 'time', 'traveller', '(for', 'so', 'it', 'will', 'be', 'convenient', 'to']
tokens 12 [['The', 'Time', 'Traveller', '(for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him)'], ['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us.', 'His', 'grey', 'eyes', 'shone', 'and'], ['twinkled,', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated.', 'The'], ['fire', 'burned', 'brightly,', 'and', 'the', 'soft', 'radiance', 'of', 'the', 'incandescent'], ['lights', 'in', 'the', 'lilies', 'of', 'silver', 'caught', 'the', 'bubbles', 'that', 'flashed', 'and'], ['passed', 'in', 'our', 'glasses.', 'Our', 'chairs,', 'being', 'his', 'patents,', 'embraced', 'and'], ['caressed', 'us', 'rather', 'than', 'submitted', 'to', 'be', 'sat', 'upon,', 'and', 'there', 'was', 'that'], ['luxurious', 'after-dinner', 'atmosphere', 'when', 'thought', 'roams', 'gracefully'], ['free', 'of', 'the', 'trammels', 'of', 'precision.', 'And', 'he'

In [27]:

def count_corpus(tokens) -> collections.Counter:
    """统计词频；支持二维 tokens（行 → 词元列表）。"""
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # tokens = [t for line in tokens for t in line]
        # tokens = [ token for line in tokens for token in line ]
        tokens = [ token for token in tokens]
    return collections.Counter(tokens)

class Vocabulary(object):
    """与课本一致的词表实现"""
    unk_token = "<unk>"
    def __init__(self, tokens=None, min_freq=1, reserved_tokens=None):
        tokens = tokens or []
        reserved_tokens = reserved_tokens or []
        # 计算词频，按从大到小排序
        token_corpus = count_corpus(tokens)
        self._token_freqs = sorted(token_corpus.items(), key = lambda x: x[1], reverse=True)
        # 过滤小于 min_freq 的词
        reserved_tokens = [res.lower() for res in reserved_tokens if res.lower() != Vocabulary.unk_token] # 过滤 Vocabulary.unk_token
        self.idx_to_token = [Vocabulary.unk_token] + reserved_tokens
        self.token_to_idx = {token: index for index, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break # 因为 _token_freqs 已经是从大到小了，因此只要出现了不满足的情况就说明，当前这个token到以后的词都小于了
            self.idx_to_token.append(token)
            self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.token_to_idx)
    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(t) for t in tokens]
    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[i] for i in indices]

    @property
    def unk(self):
        return 0
    @property
    def token_freqs(self):
        return self._token_freqs

vocab = Vocabulary(tokens=tokens, min_freq=1, reserved_tokens=["<unk>", "<PAD>", "<bos>", "<EoS>"])
print(len(vocab), vocab.idx_to_token[:10])

68 ['<unk>', '<pad>', '<bos>', '<eos>', 'us', 'sat', 'thought', 'time', 'traveller', '(for']


In [55]:
# 1、实现 one-hot编码
# 现在我有一个vocab已经构建好了，如何基于此实现one-hot编码？
class OneHotEncoder():
    def __init__(self, vocab: Vocabulary):
        self.vocab = vocab
        self.default_embedding = torch.zeros(len(vocab), dtype=torch.long)
        self.embeddings = []
        for token, idx in self.vocab.token_to_idx.items():
            tmp = self.default_embedding.clone()
            tmp[idx] = 1
            self.embeddings.append(tmp)

    def __len__(self):
        return len(self.vocab)
    def __getitem__(self, indices):
        if isinstance(indices, int):
            return self.embeddings[indices]
        if isinstance(indices, slice):
            return self.embeddings[indices]
        if isinstance(indices, (list, tuple)):
            return [self.embeddings[idx] for idx in indices]
        raise TypeError(f"Unsupported index type: {type(indices)}")

onehot = OneHotEncoder(vocab)
print(onehot, len(onehot))
print(onehot[:5])

corpus = ['us', 'sat', 'thought', 'time', 'traveller', 'time']
token_ids = vocab[corpus]
print(f"token_ids: {token_ids}")
print(f"onehot: {onehot[token_ids]}")


# 使用 torch 里面自带的 one-hot 方法
# One-Hot 编码
encoded = F.one_hot(
    torch.tensor(token_ids, dtype=torch.long),
    num_classes=len(vocab)
)

print(encoded)


<__main__.OneHotEncoder object at 0x1261f66c0> 68
[tensor([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), tensor([0, 0, 0, 0, 1, 0, 0, 

In [59]:
# 2、BoW（Bag of Words）：用词频表示一篇文档，忽略词序；
# 现在我有一个vocab已经构建好了，如何基于此实现 BoW（Bag of Words） 编码？
# BoW：一段文本对应一个向量，向量每个位置表示对应 Token 在文本中出现的次数。
"""
vocab = {"I": 0, "love": 1, "machine": 2, "learning": 3, "AI": 4}
corpus = [
    ["I", "love", "machine", "learning"],
    ["I", "love", "AI", "AI"]
]

两个bow分别对应的嵌入为：对应索引位置出现的次数
bow_embedding1 = [1, 1, 1, 1, 0]
bow_embedding2 = [1, 1, 0, 0, 2]

"""

print()
class BoWEncoder():
    def __init__(self, vocab: Vocabulary):
        self.vocab = vocab

    def __len__(self):
        return len(self.vocab)

    def __getitem__(self, token_ids):
        token_ids = token_ids or []
        tmp = torch.zeros(len(self.vocab), dtype=torch.long)
        for token in token_ids:
            tmp[token] += 1
        return tmp

corpus = [
    ["I", "love", "machine", "learning"],
    ["I", "love", "AI", "AI"]
]

tokens_test = ["I", "love", "machine", "learning", "AI"]
vocab_test = Vocabulary(tokens_test)
bow_test = BoWEncoder(vocab_test)
token_ids_test = vocab_test[corpus[0]]
print(f"token_ids_test: {token_ids_test}")
print(f"bow_test: {bow_test[token_ids_test]}")
token_ids_test = vocab_test[corpus[1]]
print(f"token_ids_test: {token_ids_test}")
print(f"bow_test: {bow_test[token_ids_test]}")


bow = BoWEncoder(vocab)
corpus = ['us', 'sat', 'thought', 'time', 'traveller', 'time']
token_ids = vocab[corpus]
print(f"token_ids: {token_ids}")
print(f"bow: {bow[token_ids]}")



token_ids_test: [1, 2, 3, 4]
bow_test: tensor([0, 1, 1, 1, 1, 0])
token_ids_test: [1, 2, 5, 5]
bow_test: tensor([0, 1, 1, 0, 0, 2])
token_ids: [4, 5, 6, 7, 8, 7]
bow: tensor([0, 0, 0, 0, 1, 1, 1, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [62]:
# 3、N-gram：把连续的 n 个 token 作为特征，保留有限的局部顺序；
# N-gram 的核心思想是：将文本中连续出现的 N 个 Token 组合成一个特征，用来保留局部词语顺序信息。
# 现在我有一个vocab已经构建好了，如何基于此实现 N-gram 编码？
from collections import Counter

def get_ngrams(tokens, n=2):
    if n <= 0:
        raise ValueError("n must be greater than 0")

    return [
        tuple(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    ]


sentence = ["I", "love", "machine", "learning"]

print(get_ngrams(sentence, n=2))
print(get_ngrams(sentence, n=3))

class NgramVocabulary:
    def __init__(self, corpus, n=2):
        self.n = n

        # 统计 N-gram 频率
        counter = Counter()

        for tokens in corpus:
            ngrams = get_ngrams(tokens, n)
            counter.update(ngrams)

        # 构建 N-gram 到 ID 的映射
        self.ngram_to_idx = {
            ngram: idx
            for idx, ngram in enumerate(counter.keys())
        }

    def __len__(self):
        return len(self.ngram_to_idx)

    def __getitem__(self, ngrams):
        if isinstance(ngrams, tuple):
            return self.ngram_to_idx[ngrams]

        return [self.ngram_to_idx[ngram] for ngram in ngrams]

corpus = [
    ["I", "love", "machine", "learning"],
    ["I", "love", "AI", "AI"]
]

ngram_vocab = NgramVocabulary(corpus, n=2)

print(ngram_vocab.ngram_to_idx)
print(len(ngram_vocab))

import torch


class NgramEncoder:
    def __init__(self, ngram_vocab):
        self.ngram_vocab = ngram_vocab

    def __len__(self):
        return len(self.ngram_vocab)

    def __getitem__(self, tokens):
        # 提取 N-gram
        ngrams = get_ngrams(tokens, self.ngram_vocab.n)

        # 创建词频向量
        encoding = torch.zeros(len(self), dtype=torch.long)

        # 统计每个 N-gram 出现次数
        for ngram in ngrams:
            idx = self.ngram_vocab.ngram_to_idx.get(ngram)

            if idx is not None:
                encoding[idx] += 1

        return encoding


corpus = [
    ["I", "love", "machine", "learning"],
    ["I", "love", "AI", "AI"]
]

ngram_vocab = NgramVocabulary(corpus, n=2)

encoder = NgramEncoder(ngram_vocab)

encoded = torch.stack([
    encoder[tokens]
    for tokens in corpus
])

print("N-gram vocab:", ngram_vocab.ngram_to_idx)
print("Encoded:", encoded)
print("Shape:", encoded.shape)

[('I', 'love'), ('love', 'machine'), ('machine', 'learning')]
[('I', 'love', 'machine'), ('love', 'machine', 'learning')]
{('I', 'love'): 0, ('love', 'machine'): 1, ('machine', 'learning'): 2, ('love', 'AI'): 3, ('AI', 'AI'): 4}
5
N-gram vocab: {('I', 'love'): 0, ('love', 'machine'): 1, ('machine', 'learning'): 2, ('love', 'AI'): 3, ('AI', 'AI'): 4}
Encoded: tensor([[1, 1, 1, 0, 0],
        [1, 0, 0, 1, 1]])
Shape: torch.Size([2, 5])


In [63]:
# 实际nlp中使用ngram
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "I love machine learning",
    "I love AI AI"
]

vectorizer = CountVectorizer(
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(corpus)

print(vectorizer.get_feature_names_out())
print(X.toarray())

['ai' 'ai ai' 'learning' 'love' 'love ai' 'love machine' 'machine'
 'machine learning']
[[0 0 1 1 0 1 1 1]
 [2 1 0 1 1 0 0 0]]


In [65]:
# 4、TF-IDF：降低整个语料中普遍出现的词的权重，突出文档特有词；
# 现在我有一个vocab已经构建好了，如何基于此实现 TF-IDF 编码？

import torch
import math


class TFIDFEncoder:
    def __init__(self, vocab: Vocabulary, corpus):
        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.doc_count = len(corpus)

        # 1. 计算 DF
        self.doc_freq = torch.zeros(
            self.vocab_size,
            dtype=torch.long
        )

        for doc in corpus:
            for token in set(doc):
                idx = self.vocab[token]
                self.doc_freq[idx] += 1

        # 2. 计算 IDF
        self.idf = torch.zeros(
            self.vocab_size,
            dtype=torch.float32
        )

        for idx in range(self.vocab_size):
            df = self.doc_freq[idx]
            if df > 0:
                self.idf[idx] = math.log(
                    self.doc_count / df
                )

    def __len__(self):
        return self.vocab_size

    def compute_tf(self, tokens):
        tf = torch.zeros(
            self.vocab_size,
            dtype=torch.float32
        )

        for token in tokens:
            idx = self.vocab[token]
            tf[idx] += 1

        if len(tokens) > 0:
            tf /= len(tokens)

        return tf

    def __getitem__(self, tokens):
        tf = self.compute_tf(tokens)

        return tf * self.idf


# 准备语料
corpus = [
    ["I", "love", "machine", "learning"],
    ["I", "love", "AI", "AI"],
    ["I", "love", "deep", "learning"]
]

# 创建 TF-IDF 编码器
tfidf = TFIDFEncoder(vocab, corpus)

# 编码整个语料
encoded = torch.stack([
    tfidf[doc]
    for doc in corpus
])

print("DF:", tfidf.doc_freq)
print("IDF:", tfidf.idf)
print("TF-IDF:")
print(encoded)
print("Shape:", encoded.shape)


DF: tensor([11,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])
IDF: tensor([-1.2993,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000])
T

In [66]:

import jieba
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "我喜欢自然语言处理",
    "自然语言处理很有趣",
    "机器学习可以分析文本",
]

def chinese_tokenizer(text):
    return [word for word in jieba.lcut(text) if word.strip()]

vectorizer = TfidfVectorizer(
    tokenizer=chinese_tokenizer,
    token_pattern=None,
)
X = vectorizer.fit_transform(documents)
print(vectorizer.get_feature_names_out())
print(X.toarray().round(3))


Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/62/z650ynp965s9rqn86r5kl6kr0000gn/T/jieba.cache
Loading model cost 0.487 seconds.
Prefix dict has been built successfully.


['分析' '可以' '喜欢' '处理' '学习' '很' '我' '文本' '有趣' '机器' '自然语言']
[[0.    0.    0.563 0.428 0.    0.    0.563 0.    0.    0.    0.428]
 [0.    0.    0.    0.428 0.    0.563 0.    0.    0.563 0.    0.428]
 [0.447 0.447 0.    0.    0.447 0.    0.    0.447 0.    0.447 0.   ]]


In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "machine learning is useful",
    "deep learning is powerful",
    "natural language processing",
]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)

print(vectorizer.get_feature_names_out())
print(X.shape)
print(X.toarray().round(3))

['deep' 'is' 'language' 'learning' 'machine' 'natural' 'powerful'
 'processing' 'useful']
(3, 9)
[[0.    0.428 0.    0.428 0.563 0.    0.    0.    0.563]
 [0.563 0.428 0.    0.428 0.    0.    0.563 0.    0.   ]
 [0.    0.    0.577 0.    0.    0.577 0.    0.577 0.   ]]
